In [3]:
import os
import copy
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import itertools
import warnings

warnings.filterwarnings('ignore')

# ==========================================
# 1. 全局配置与路径
# ==========================================
DEBUG_MODE = False  # 调参阶段建议设为 False
DATA_PATH = '../data/03_final_feature_matrix.csv'
MODEL_DIR = '../models/grid_search_baseline/'
RESULT_DIR = '../results/'
LOG_FILE = os.path.join(RESULT_DIR, 'grid_search_results_baseline.csv')

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(RESULT_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ==========================================
# 2. 网格搜索空间定义 (Grid Search Space)
# ==========================================
param_grid = {
    'hidden_size': [64, 128],
    'dropout': [0.3, 0.5],
    'batch_size': [32, 64],
    'lr': [0.001, 0.0005],
    'feature_group': ['G1', 'G2', 'G3', 'G3-M']
}

# 如果是调试模式，极大压缩搜索空间
if DEBUG_MODE:
    param_grid = {
        'hidden_size': [64],
        'dropout': [0.3],
        'batch_size': [32],
        'lr': [0.001],
        'feature_group': ['G1']
    }

# ==========================================
# 3. 数据集与模型定义 (保持结构对齐)
# ==========================================
class CryptoVolatilityDataset(Dataset):
    def __init__(self, feature_matrix, target_array, lookback_window=7):
        self.features = feature_matrix
        self.targets = target_array
        self.window = lookback_window
        
    def __len__(self):
        return len(self.features) - self.window + 1
        
    def __getitem__(self, idx):
        x = self.features[idx : idx + self.window]
        y = self.targets[idx + self.window - 1]
        return torch.FloatTensor(x), torch.FloatTensor([y])

class BaselineLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, dropout_p):
        super(BaselineLSTM, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers=1, batch_first=True)
        self.dropout = nn.Dropout(dropout_p)
        self.fc = nn.Sequential(
            nn.Linear(hidden_size, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )
        
    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.dropout(out[:, -1, :])
        return self.fc(out)

# ==========================================
# 4. 核心训练函数 (含防御机制)
# ==========================================
def train_and_validate(model, train_loader, val_loader, params, config_str):
    criterion = nn.MSELoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=params['lr'], weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', factor=0.5, patience=7)
    
    best_val_loss = float('inf')
    best_model_wts = None
    patience, epochs_no_improve = 15, 0
    max_epochs = 5 if DEBUG_MODE else 200
    
    for epoch in range(max_epochs):
        model.train()
        for x_batch, y_batch in train_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            loss = criterion(model(x_batch), y_batch)
            loss.backward()
            optimizer.step()
            
        model.eval()
        val_losses = []
        with torch.no_grad():
            for x_v, y_v in val_loader:
                x_v, y_v = x_v.to(device), y_v.to(device)
                val_losses.append(criterion(model(x_v), y_v).item())
        
        avg_val_loss = np.mean(val_losses)
        scheduler.step(avg_val_loss)
        
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_model_wts = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            
        if epochs_no_improve >= patience:
            break
            
    model.load_state_dict(best_model_wts)
    return model, best_val_loss

# ==========================================
# 5. 网格搜索主逻辑
# ==========================================
def run_grid_search():
    df = pd.read_csv(DATA_PATH)
    all_cols = list(df.columns)
    base_features = [c for c in all_cols if c not in ['timestamp', 'target_sigma_t_plus_1', 'finbert_sentiment_score', 'llm_composite_score', 'sentiment_class', 'action_class', 'action_score']]
    
    # 特征组映射
    fg_map = {
        'G1': base_features,
        'G2': base_features + ['finbert_sentiment_score'],
        'G3': base_features + ['llm_composite_score'],
        'G3-M': base_features + ['sentiment_class', 'action_class', 'action_score']
    }

    # 生成所有参数组合
    keys, values = zip(*param_grid.items())
    combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]
    
    print(f"开始网格搜索，共计 {len(combinations)} 组实验...")

    for i, params in enumerate(combinations):
        config_id = f"LSTM_{params['feature_group']}_h{params['hidden_size']}_dp{params['dropout']}_bs{params['batch_size']}_lr{params['lr']}"
        print(f"[{i+1}/{len(combinations)}] 正在训练: {config_id}")
        
        # 1. 准备数据
        f_cols = fg_map[params['feature_group']]
        train_end, val_end = int(len(df)*0.7), int(len(df)*0.8)
        
        train_ds = CryptoVolatilityDataset(df.iloc[:train_end][f_cols].values, df.iloc[:train_end]['target_sigma_t_plus_1'].values)
        val_ds = CryptoVolatilityDataset(df.iloc[train_end:val_end][f_cols].values, df.iloc[train_end:val_end]['target_sigma_t_plus_1'].values)
        test_ds = CryptoVolatilityDataset(df.iloc[val_end:][f_cols].values, df.iloc[val_end:]['target_sigma_t_plus_1'].values)
        
        train_loader = DataLoader(train_ds, batch_size=params['batch_size'], shuffle=True)
        val_loader = DataLoader(val_ds, batch_size=params['batch_size'], shuffle=False)
        test_loader = DataLoader(test_ds, batch_size=params['batch_size'], shuffle=False)
        
        # 2. 训练与早停
        torch.cuda.empty_cache()
        model = BaselineLSTM(len(f_cols), params['hidden_size'], params['dropout']).to(device)
        model, _ = train_and_validate(model, train_loader, val_loader, params, config_id)
        
        # 3. 测试集评估
        model.eval()
        preds, targets = [], []
        with torch.no_grad():
            for x_t, y_t in test_loader:
                preds.extend(model(x_t.to(device)).cpu().numpy())
                targets.extend(y_t.numpy())
        
        preds, targets = np.array(preds).flatten(), np.array(targets).flatten()
        rmse = np.sqrt(mean_squared_error(targets, preds))
        mae = mean_absolute_error(targets, preds)
        r2 = r2_score(targets, preds)
        
        # 4. 记录结果与保存模型
        result_entry = {**params, 'RMSE': rmse, 'MAE': mae, 'R2': r2, 'Timestamp': time.strftime("%Y-%m-%d %H:%M:%S")}
        res_df = pd.DataFrame([result_entry])
        res_df.to_csv(LOG_FILE, mode='a', header=not os.path.exists(LOG_FILE), index=False)
        
        torch.save(model.state_dict(), os.path.join(MODEL_DIR, f"best_{config_id}.pth"))

if __name__ == '__main__':
    run_grid_search()

开始网格搜索，共计 64 组实验...
[1/64] 正在训练: LSTM_G1_h64_dp0.3_bs32_lr0.001
[2/64] 正在训练: LSTM_G2_h64_dp0.3_bs32_lr0.001
[3/64] 正在训练: LSTM_G3_h64_dp0.3_bs32_lr0.001
[4/64] 正在训练: LSTM_G3-M_h64_dp0.3_bs32_lr0.001
[5/64] 正在训练: LSTM_G1_h64_dp0.3_bs32_lr0.0005
[6/64] 正在训练: LSTM_G2_h64_dp0.3_bs32_lr0.0005
[7/64] 正在训练: LSTM_G3_h64_dp0.3_bs32_lr0.0005
[8/64] 正在训练: LSTM_G3-M_h64_dp0.3_bs32_lr0.0005
[9/64] 正在训练: LSTM_G1_h64_dp0.3_bs64_lr0.001
[10/64] 正在训练: LSTM_G2_h64_dp0.3_bs64_lr0.001
[11/64] 正在训练: LSTM_G3_h64_dp0.3_bs64_lr0.001
[12/64] 正在训练: LSTM_G3-M_h64_dp0.3_bs64_lr0.001
[13/64] 正在训练: LSTM_G1_h64_dp0.3_bs64_lr0.0005
[14/64] 正在训练: LSTM_G2_h64_dp0.3_bs64_lr0.0005
[15/64] 正在训练: LSTM_G3_h64_dp0.3_bs64_lr0.0005
[16/64] 正在训练: LSTM_G3-M_h64_dp0.3_bs64_lr0.0005
[17/64] 正在训练: LSTM_G1_h64_dp0.5_bs32_lr0.001
[18/64] 正在训练: LSTM_G2_h64_dp0.5_bs32_lr0.001
[19/64] 正在训练: LSTM_G3_h64_dp0.5_bs32_lr0.001
[20/64] 正在训练: LSTM_G3-M_h64_dp0.5_bs32_lr0.001
[21/64] 正在训练: LSTM_G1_h64_dp0.5_bs32_lr0.0005
[22/64] 正在训练: LSTM_G2_h64